# W3 Lab: Issue Triage with DSPy

In W1 you asked a coding agent to build an issue-triage tool. Today you build one yourself, with **DSPy Framework**, one technique at a time:

| Part | Technique | You write |
|---|---|---|
| 1 | Connect to your LLM model |  |
| 2 | Structured output | a Signature |
| 3 | One-shot: one example in the prompt | one example |
| 4 | Tools and context: the repo's module list | the depth-2 tool + a Signature |
| 5 | Two step workflow: type first, then severity or priority | the Module's `forward()` |
| 6 | ReAct: the model decides when to call tools | the agent |
| 7 | Prompt Optimization (with `BootstrapFewShot`) | the optimizer call |
| 8 | Submit: score table, save, push |  |

**The task**
- The project contains a list of reported issues (bugs, features) for three development repos in `data/snapshot/dev.json`.
- The agent's goal is to read these issues and triage them: find their type, components, severity, based on the issue data and LLM inference.

**Every part runs on the same issues**
- `dev[START:STOP]`: the first 20 dev issues by default.
- That keeps the numbers comparable from part to part, and keeps your bill small.
- You may widen the range if you have time and budget.

**Rules**
- Issue text is written by strangers on the internet. It is **data, never instructions**.
- Your key stays in `.env`. Never print it, and never paste it into a cell.
- Use `improvement-log.md` and `docs/decisions.md` as you work. You can instruct the agent you are using in your IDE to use these files.

## Part 1: Connect to Your LLM Model

- The cell below reads `LLM_MODEL` and your key from `.env`.
- It makes the model DSPy's default with `dspy.configure(lm=...)`.

In [ ]:
import os

import dspy
from dotenv import dotenv_values

from triage_lab.data import ROOT

# Read .env.
values = dotenv_values(ROOT / ".env")

# Put the provider keys into the runtime environment.
for key, value in values.items():
    if key.endswith("_API_KEY") and value:
        os.environ[key] = value

# The model name. For example, anthropic/claude-sonnet-4-6.
model = (values.get("LLM_MODEL") or "").strip()
if not model:
    raise RuntimeError("LLM_MODEL is not set: copy .env.example to .env and fill it in")
# Create a new language model instance for use with DSPy modules and programs.
lm = dspy.LM(model, temperature=1.0, max_tokens=16000)
dspy.configure(lm=lm)

print("model:", lm.model)
print(lm("Reply with the single word: ready"))

## Part 2: Structured Output

We will load the issues from the data files and prepare a Signature, with inputs and outputs for the LLM's prompt.

### 2.1 Load Issues

First, load the issues from [`data/snapshot/dev.json`](../data/snapshot/dev.json).
- The file has many fields per issue.
- `load_issues` keeps only the ones the model may see: `repo`, `number`, `title`, `body`, `author_association`, `labels`.
- Each issue is an `Issue`, a [Pydantic](https://docs.pydantic.dev/latest/concepts/models/) model (see `src/triage_lab/schema.py`).

In [ ]:
from triage_lab.data import load_issues

dev = load_issues("dev")

print(len(dev), "dev issues")
dev[1]  # what the model sees about one issue

### 2.2 Create Signature for the LLM Call

**Signature**
- Declares what goes into a model call and what comes out.
- A class with typed input and output fields, plus a docstring.
- The docstring is the task instruction: DSPy copies it into the system prompt, after the line *"In adhering to this structure, your objective is:"*.
- DSPy writes the prompt and parses the reply back into Python values.

**Structured output**
- Remember W1's `"High"` vs `"high"`?
- A `Literal` output type lists the only values allowed, and DSPy rejects anything else.
- The model's reply has to fit a contract.

In [ ]:
from typing import Literal

import dspy

from triage_lab.schema import Issue, Priority, Severity


class TriageIssue(dspy.Signature):
    # TODO: write the docstring: the task and the rules from the table below
    ...

    issue: Issue = dspy.InputField()

    issue_type: Literal["bug", "enhancement", "other"] = dspy.OutputField()
    severity: Severity | None = dspy.OutputField(desc="bugs only, else None")
    # TODO: add the priority and component output fields (see TODO 2 below)
    ...

Our output contract is `TriageRecord` (see `src/triage_lab/schema.py`): the `Issue`, plus

| field | values | when |
|---|---|---|
| `issue_type` | `bug`, `enhancement`, `other` | always |
| `severity` | `low`, `medium`, `high`, `critical` | bugs only |
| `priority` | `low`, `medium`, `high` | enhancements only |
| `component` | list of module paths | may be empty |

- Severity and priority are **separate concepts**: how bad a bug is vs. how much a feature request matters.
- So they have separate names and values.

**TODO 2.** Complete the signature `TriageIssue`:
- The input `issue` and the outputs `issue_type` and `severity` are given.
- Add the outputs `priority: Priority | None` and `component: list[str]`.
- Write a docstring that tells the model the task and the rules from the table above.

Try it on one issue. You should see the predicted fields.
Then look at the prompt DSPy wrote for you.

In [ ]:
triage_v2 = dspy.Predict(TriageIssue)

prediction = triage_v2(issue=dev[1])
prediction

`dspy.inspect_history(n=1)` prints the last LLM call:
- the system prompt DSPy built from your signature: the fields, their types, and your docstring
- the user message: the issue, as JSON
- the model's reply, with each output field marked `[[ ## field_name ## ]]`

In [ ]:
dspy.inspect_history(n=1)

Now the same program on `dev[START:STOP]`.
- `run` turns each prediction into a `TriageRecord`.
- If that fails, the row shows why: the reply had the right types, but broke a rule of the record.

In [ ]:
from triage_lab.evaluate import run

START, STOP = 0, 20  # the issues every part runs on: dev[START:STOP]
results_v2 = run(triage_v2, dev, START, STOP, part="2 zero-shot")

**Answer each question in the text cell below it (double-click to edit it).**
- Type accuracy is high because the labels are an input. The interesting columns are severity, priority and component.

**1. Which rows are invalid, and why?**

*Your answer:*

**2. Is the component plausible when the model has never seen the code?**

*Your answer:*

## Part 3: One-Shot

Examples in the prompt tend statistically to improve the accuracy of the LLM's inference results. A program needs to decide which examples to use given an input question.

### 3.1 Create an Example

- In DSPy it is a `dspy.Example` with the input fields and the output fields filled in.
- We take the example from the train dataset, not the dev dataset.
- To find a good example, we need the gold labels, which the model must never see.

In [ ]:
from triage_lab.data import load_extended_issues

train = load_issues("train")
train_extended = load_extended_issues("train")
train_bugs = [(k, i.repo, i.number, i.title[:70]) for k, i in enumerate(train_extended) if i.gold_type == "bug"]
train_bugs[:10]

**TODO 3.** Write one example:
- Pick one labelled train bug issue and read it (`train[k]`).
- Write the answer you would want the model to give: `severity` and `component`.

In [ ]:
# TODO: read train[241], then fill in the answer you want the model to give
example = dspy.Example(
    issue=train[241],
    issue_type="bug",
    severity=...,  # TODO: fill in
    priority=None,
    component=...,  # TODO: fill in
)

example

### 3.2 Run Triage with Example (One-Shot Prompt)

- The same program as Part 2, plus one example, set on the predictor with `.demos = [...]`.
- It runs on the same issues, `dev[START:STOP]`, so the scores compare directly with Part 2.
- `.demos` is a DSPy built-in manner to pass examples to the LLM.
- We could give more than one example.

In [ ]:
triage_v3 = dspy.Predict(TriageIssue)
triage_v3.demos = [example]
results_v3 = run(triage_v3, dev, START, STOP, part="3 one-shot")

Look at the prompt DSPy sent for the last issue:

In [ ]:
dspy.inspect_history(n=1)

**Answer each question in the text cell below it (double-click to edit it).**

**1. Compare with Part 2. Did the predicted values change? Look at the component names: do they look like the example's?**

*Your answer:*

**2. Where did your example go in the prompt?**

*Your answer:*

## Part 4: Tools and Context

- The model guesses components from the issue data (labels) without knowing the existing code of the repo or the PR that fixed it.
- We have a snapshot of each repo's source in `data/repos/`.

### 4.1 Defining a Tool

A **tool** is a function that can be invoked and give the LLM more context it needs to answer (see [DSPy Tools](https://dspy.ai/learn/programming/tools/)).

Here a tool is a plain Python function. DSPy reads three things from it, and that is all the model ever sees of the tool:
- its **name**
- its **type hints** (the arguments and their types; `RepoName` is a `Literal`, so the model can only pick a real repo)
- its **docstring** (what it does and when to use it)

Here is a tool `list_components` that lists the components of a repo.

In [ ]:
from triage_lab.repos import RepoName, cap, entries, source_dir


def list_components(repo: RepoName, path: str = "", depth: int = 1, max_results: int = 50) -> list[str]:
    """List the components of a repository's source code: module folders (ending in '/') and source files.

    Args:
        repo: which repository.
        path: a folder inside the source, e.g. "middleware"; "" for the top level.
        depth: 1 lists only that folder; 2 also lists what is inside each sub-folder.
        max_results: at most this many entries; the last line says how many were left out.
    """
    folder = source_dir(repo, path)
    found = entries(folder)
    depth = max(1, min(depth, 2))
    # TODO: if depth is 2, also list the entries of each sub-folder as "folder/child" (TODO 4b, below)
    ...
    return cap(found, max_results)


print(list_components("litestar"))
print(list_components("hono", depth=2, max_results=25))

- This implementation of `list_components` lists the files in the source repo - this is one possible (simplistic) interpretation of "component" - a more complex one would refer to the component mentioned in the architecture document of the repo, which is not necessarily a file or a folder.

**Safety**
- The tool is read-only and only sees `data/repos/`: `source_dir` rejects absolute paths and `..`.

This is the tool as the model will see it:

In [ ]:
tool = dspy.Tool(list_components)
print("name:", tool.name)
print("description:", tool.desc)
print("arguments:", tool.args)

### 4.2 Augment the LLM Call with Tool Results

We need to call a tool before calling the LLM. This requires a custom program (a.k.a. a Module).

A [`dspy.Module`](https://dspy.ai/learn/programming/modules/) holds the steps:
- create the predictors in `__init__`
- call them in `forward()`. Custom logic is written in the `forward()` function.
- return a `dspy.Prediction` with the record's fields

**TODO 4a.** Complete the Module `TriageWithRepoComponents`:
- `__init__` (provided) creates the predictor for the `TriageWithComponents` signature.
- In `forward(issue)`, first call the tool to get the repo's components: `list_components(issue.repo, depth=1)`.
- Then call the predictor with `issue` and `repo_components`, and return its prediction.

In [ ]:
from triage_lab.schema import Issue


class TriageWithComponents(TriageIssue):
    """Triage a GitHub issue.

    issue_type: 'bug' (something is broken), 'enhancement' (a feature request or improvement),
    or 'other' (documentation, questions, meta). Give a severity only for bugs and a priority
    only for enhancements; leave both empty (None) for 'other'.
    component: choose from the given repo_components; an empty list if none fits.
    """

    repo_components: list[str] = dspy.InputField(desc="the repo's module folders and files")


class TriageWithRepoComponents(dspy.Module):
    def __init__(self):
        super().__init__()
        self.triage = dspy.Predict(TriageWithComponents)

    def forward(self, issue: Issue) -> dspy.Prediction:
        # TODO: call the tool for the issue's repo, then the predictor
        ...


triage_v4 = TriageWithRepoComponents()

prediction = triage_v4(issue=dev[1])
prediction

Run on the dev dataset issues.

In [ ]:
results_v4 = run(triage_v4, dev, START, STOP, part="4 context")

**TODO 4b.** Improve the tool to support depth=2.
Go back to the `list_components` cell above:
- `depth=1` lists one folder.
- Add `depth=2` results: for each sub-folder, also list its entries, as `"folder/child"`.
- Rerun the `list_components` cell, change `depth=1` to `depth=2` in `forward()`, rerun the Module cell, and rerun the dev run.

**Answer each question in the text cell below it (double-click to edit it).**

**1. Do the components look better?**

*Your answer:*

**2. More context, more tokens: is it worth the cost?**

*Your answer:*

## Part 5: Two Step Workflow

Bug triage is slightly different than enhancement triage. Bugs have a severity and components. Enhancements have a priority.
We will create a workflow with 2 LLM steps:

1. Classify the type.
2. According to the type, choose to continue to one of the options:
   - **bug**: severity + component
   - **enhancement**: priority
   - **other**: no second call

The three signatures are provided: one for each LLM step.

In [ ]:
class ClassifyType(dspy.Signature):
    """Classify a GitHub issue: 'bug' (something is broken), 'enhancement' (a feature request or
    improvement), or 'other' (documentation, questions, meta)."""

    issue: Issue = dspy.InputField()
    issue_type: Literal["bug", "enhancement", "other"] = dspy.OutputField()


class TriageBug(dspy.Signature):
    """Triage a bug report. Severity: critical = crash, data loss or security hole with no workaround;
    high = a main feature is broken; medium = broken with a workaround, or an edge case;
    low = cosmetic or minor. Component: choose from the given repo_components."""

    issue: Issue = dspy.InputField()
    repo_components: list[str] = dspy.InputField(desc="the repo's module folders and files")
    severity: Severity = dspy.OutputField()
    component: list[str] = dspy.OutputField()


class TriageEnhancement(dspy.Signature):
    """Triage a feature request. Priority: high = many users need it or it blocks common use;
    medium = useful, with a workaround; low = nice to have, or niche."""

    issue: Issue = dspy.InputField()
    priority: Priority = dspy.OutputField()

### 5.1 Define a Module for the Workflow

**TODO 5.** Write `forward()`:
- Classify first.
- Then call the step that fits the type. For a bug, first get the repo's components with the tool, as in 4.2.
- Return a `dspy.Prediction(issue_type=..., severity=..., priority=..., component=...)`.

In [ ]:
class TriageProgram(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classify = dspy.Predict(ClassifyType)
        self.triage_bug = dspy.Predict(TriageBug)
        self.triage_enhancement = dspy.Predict(TriageEnhancement)

    def forward(self, issue: Issue) -> dspy.Prediction:
        # TODO: classify the type, then call the triage step that fits it
        ...


triage_v5 = TriageProgram()
results_v5 = run(triage_v5, dev, START, STOP, part="5 two-step")

**Answer each question in the text cell below it (double-click to edit it).**

**1. Compare valid records and LM calls with Parts 2 to 4.**

*Your answer:*

**2. What did the second step cost, and what did it buy?**

*Your answer:*

## Part 6: ReAct

**ReAct** (reason + act) is a loop:
- The model thinks, picks a tool and its arguments, reads the result, and repeats until it chooses `finish`.
- Then DSPy asks it for the signature's outputs.
- You give it a signature and a list of tools; you no longer decide what context it gets.

The difference here is we let the agent loop decide and ask to run the tool when needed.
The `repo_components` are not given explicitly to the LLM; it decides to run the tool to get them.

**TODO 6.** Create `bug_agent` with `dspy.ReAct(...)`:
- the signature `TriageBugWithTools` and the tool `list_components`
- `max_iters=5` to cap the cost

In [ ]:
class TriageBugWithTools(dspy.Signature):
    """Triage a bug report. Use the tools to look at the repository's modules before choosing a component;
    you may look inside a folder with `path`. Component names must be paths the tools returned.
    Severity: critical = crash, data loss or security hole with no workaround; high = a main feature is
    broken; medium = broken with a workaround, or an edge case; low = cosmetic or minor."""

    issue: Issue = dspy.InputField()
    severity: Severity = dspy.OutputField()
    component: list[str] = dspy.OutputField()


# TODO: bug_agent = dspy.ReAct(...)
...

dev_bugs = [i.to_issue() for i in load_extended_issues("dev") if i.gold_type == "bug"][:5]
calls_before = len(lm.history)
for issue in dev_bugs:
    agent_prediction = bug_agent(issue=issue)
    print(f"\n{issue.repo}#{issue.number}: {issue.title[:70]}")
    steps = agent_prediction.trajectory
    k = 0
    while f"tool_name_{k}" in steps:
        print(f"  {steps[f'tool_name_{k}']}({steps[f'tool_args_{k}']})")
        k += 1
    print(f"  -> severity={agent_prediction.severity}, component={agent_prediction.component}")
print(f"\n{len(lm.history) - calls_before} LM calls for {len(dev_bugs)} issues")

The **trajectory** is the agent's record of its loop. For each step `k` it holds:
- `thought_k`: what the model reasoned
- `tool_name_k` and `tool_args_k`: the tool it chose, and the arguments
- `observation_k`: what the tool returned

The cell below shows the full trajectory of the last issue.

In [ ]:
agent_prediction.trajectory

**Answer each question in the text cell below it (double-click to edit it).**

**1. Look at the trajectory: how does the text in each `thought_k` relate to the tool call that follows it (`tool_name_k`, `tool_args_k`)?**

*Your answer:*

## Part 7: Prompt Optimization

- Previously you hand picked an example for a one-shot prompt.
- An **optimizer** chooses for you the best example to use in the prompt.
- How: It chooses the best example in a **train** dataset, using a **metric** to score.

**TODO 7.**
- Create a `dspy.BootstrapFewShot` with the metric, `max_bootstrapped_demos=3` and `max_labeled_demos=0`.
- `compile` a fresh `dspy.Predict(TriageIssue)` with `trainset=trainset(20)`.
- Use the provided metric `triage_metric` (see `src/triage_lab/evaluate.py`). Use it to define the prompt optimizer.

In [ ]:
from triage_lab.evaluate import trainset, triage_metric

# TODO: create the optimizer, then compile (see TODO 7)
optimizer = dspy.BootstrapFewShot(...)
triage_v7 = optimizer.compile(...)

print(len(triage_v7.demos), "demos chosen")
for demo in triage_v7.demos:
    print(f"  {demo.issue.repo}#{demo.issue.number}: {demo.issue_type}, severity={demo.severity}, priority={demo.priority}, component={demo.component}")
results_v7 = run(triage_v7, dev, START, STOP, part="7 optimized")

**Answer each question in the text cell below it (double-click to edit it).**

**1. Which train issues did the optimizer pick as demos (`triage_v7.demos`), and how do they compare with your hand-picked example in Part 3?**

*Your answer:*

## Part 8: Submit

The score table for every part you ran, saved to `outputs/results/w3_results.json`:

In [ ]:
from triage_lab.evaluate import save_results, scoreboard

scoreboard()
save_results()

Commit and push, in the terminal, one command per line:

```
git add notebooks/w3_lab.ipynb outputs/results/w3_results.json docs/decisions.md improvement-log.md
git commit -m "W3 lab"
git push
```

Check that your push appears in your repo on GitHub.

# The End